# 🔬 Nuclei Segmentation MoNuSeg 2018
## Final Project · Biomedical Image Processing

**Institut Teknologi Sepuluh Nopember** · Biomedical Engineering

---
### Pipeline
1. **H&E Color Deconvolution**: isolate hematoxylin (nucleus) channel
2. **CLAHE Enhancement**: adaptive local contrast normalisation
3. **Otsu Thresholding**: data-driven binary mask
4. **Morphological Cleanup**: open / close / fill holes
5. **Distance Transform + Watershed**: split touching nuclei
6. **Size Filtering**: remove spurious regions

### Reported Metrics
| Metric | Description |
|---|---|
| **IoU (Jaccard)** | Intersection / Union |
| **Dice Similarity** | 2·Intersection / (Predicted + Ground Truth/GT) |
| **Running Time (s)** | Wall-clock time per image |


In [ ]:
# Install / verify required packages  (run once)
import subprocess, sys

_pkgs = [
    "numpy", "opencv-python-headless", "scipy",
    "scikit-image", "matplotlib", "tifffile", "pandas",
]
for _p in _pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", _p, "-q"],
                   capture_output=True)

print("✓ All packages ready.")


## 1  · Setup and Imports

In [ ]:
import numpy as np
import cv2
import os, time, warnings
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from scipy import ndimage as ndi
from scipy.ndimage import distance_transform_edt, binary_fill_holes

warnings.filterwarnings("ignore")

# tifffile (preferred TIFF reader)
try:
    import tifffile
    _USE_TIFFFILE = True
except ImportError:
    _USE_TIFFFILE = False
    print("tifffile not found – falling back to cv2 for TIFF loading")

print(f"NumPy   {np.__version__}")
print(f"OpenCV  {cv2.__version__}")
print("✓ Imports complete")


In [ ]:
# GPU Detection (CuPy for distance transform acceleration)
GPU_AVAILABLE = False

try:
    import cupy as cp
    import cupyx.scipy.ndimage as cpnd
    _test = cp.zeros(1)           # allocates on GPU
    GPU_AVAILABLE = True
    _dev = cp.cuda.Device()
    _mem_gb = _dev.mem_info[1] / 1e9
    print(f"✓ GPU (CuPy) detected  |  Device {_dev.id}  |  {_mem_gb:.1f} GB VRAM")
except Exception as _e:
    print(f"✗ CuPy unavailable ({type(_e).__name__}) – running on CPU")
    print("  To enable GPU: pip install cupy-cuda12x  (match your CUDA version)")


## 2  · Configuration

In [ ]:
#  DATASET PATHS  ← adjust BASE_DIR to your local MoNuSeg2018 folder
BASE_DIR       = Path("MoNuSeg2018")
ANNOTATION_DIR = BASE_DIR / "Annotations"
TISSUE_DIR     = BASE_DIR / "Tissue Images"
OUTPUT_DIR     = Path("results"); OUTPUT_DIR.mkdir(exist_ok=True)

IMAGE_NAMES = [
    "TCGA-AR-A1AS-01Z-00-DX1",
    "TCGA-AY-A8YK-01A-01-TS1",
    "TCGA-E2-A1B5-01Z-00-DX1",
    "TCGA-RD-A8N9-01A-01-TS1",
]

#  SEGMENTATION PARAMETERS
PARAMS = {
    # Preprocessing
    "clahe_clip_limit"   : 4,    # higher  → stronger contrast boost #default 2
    "clahe_tile_size"    : (8, 8), # smaller tile → more local adaptation #default (8, 8)
    "gaussian_sigma"     : 2.0,    # smoothing sigma before thresholding #default 1.0

    # Thresholding
    # otsu_factor < 1.0  → lower threshold → more pixels as nuclei (↑ recall)
    # otsu_factor > 1.0  → higher threshold → fewer pixels (↑ precision)
    "otsu_factor"        : 0.75, #default 0.9

    # Morphology
    "open_radius"        : 2,      # removes thin noise connections #default 2
    "close_radius"       : 3,      # fills small interior gaps #default 3

    # Watershed
    "peak_min_dist"      : 5,      # minimum pixel distance between nucleus centres #default 10
    "dist_thresh_frac"   : 0.05,   # fraction of max distance for peak detection #default 0.3
                                   # lower → detect smaller/more markers

    # Size Filtering
    "min_area_px"        : 50,     # discard nuclei smaller than this (pixels²) #default 50
    "max_area_px"        : 6000,   # discard nuclei larger than this  (pixels²) #default 6000
}

print("Configuration loaded.")
print(f"  Dataset : {BASE_DIR.resolve()}")
print(f"  Output  : {OUTPUT_DIR.resolve()}")


## 3  · Ground-Truth: XML Annotation Parser

In [ ]:
def parse_xml_to_mask(xml_path: Path, image_shape: tuple) -> np.ndarray:
    """
    Convert a MoNuSeg XML annotation file to a binary ground-truth mask

    MoNuSeg XML structure (Aperio ImageScope / QuPath export):
        <Annotations>
          <Annotation ...>
            <Regions>
              <Region Id="...">
                <Vertices>
                  <Vertex X="..." Y="..."/>
                  ...
                </Vertices>
              </Region>
            </Regions>
          </Annotation>
        </Annotations>

    Parameters
    ----------
    xml_path    : Path to the .xml file
    image_shape : (H, W) or (H, W, C) of the corresponding tissue image

    Returns
    -------
    mask : np.ndarray (H, W) uint8 — 255 = nucleus, 0 = background
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    H, W = image_shape[:2]
    mask  = np.zeros((H, W), dtype=np.uint8)
    count = 0

    for region in root.iter("Region"):
        verts_el = region.find("Vertices")
        if verts_el is None:
            continue

        coords = []
        for vtx in verts_el.findall("Vertex"):
            try:
                x = int(np.clip(round(float(vtx.get("X", 0))), 0, W - 1))
                y = int(np.clip(round(float(vtx.get("Y", 0))), 0, H - 1))
                coords.append([x, y])
            except (ValueError, TypeError):
                continue

        if len(coords) >= 3:
            pts = np.array(coords, dtype=np.int32)
            cv2.fillPoly(mask, [pts], color=255)
            count += 1

    print(f"    Parsed {count:4d} nuclei  ←  {xml_path.name}")
    return mask


## 4  · H&E Colour Deconvolution

In [ ]:
# Standard H&E stain vectors  (Ruifrok and Johnston, 2001)
#_H = np.array([0.6442, 0.7166, 0.2668])   # Hematoxylin  : nuclei (blue-purple)
#_E = np.array([0.0928, 0.9541, 0.2836])   # Eosin        : cytoplasm (pink)
_H = np.array([0.651, 0.701, 0.290])   # Hematoxylin  : nuclei (blue-purple)
_E = np.array([0.216, 0.801, 0.558])   # Eosin        : cytoplasm (pink)
_R = np.cross(_H, _E)
_R /= np.linalg.norm(_R) + 1e-12          # Residual (DAB / background)

HE_MATRIX     = np.stack([_H, _E, _R], axis=0)    # shape (3, 3)
HE_MATRIX_INV = np.linalg.inv(HE_MATRIX)


def deconvolve_he(image_rgb: np.ndarray):
    """
    Separate an H&E image into individual stain concentration maps

    Parameters
    ----------
    image_rgb : np.ndarray  (H, W, 3) uint8 [0, 255]

    Returns
    -------
    H_ch : np.ndarray  (H, W) float, hematoxylin concentration (nucleus signal)
    E_ch : np.ndarray  (H, W) float, eosin concentration
    """
    img = np.clip(image_rgb.astype(np.float64) / 255.0, 1e-6, 1.0)
    OD  = -np.log10(img)                       # Optical Density  (H, W, 3)
    stains = (HE_MATRIX_INV @ OD.reshape(-1, 3).T).T   # (N, 3)
    stains = np.clip(stains, 0, None)

    h, w   = image_rgb.shape[:2]
    H_ch   = stains[:, 0].reshape(h, w)
    E_ch   = stains[:, 1].reshape(h, w)
    return H_ch, E_ch


## 5  · Preprocessing

In [ ]:
def preprocess(image_rgb: np.ndarray, params: dict = PARAMS):
    """
    Preprocess H&E image: deconvolve → normalise → CLAHE → Gaussian blur

    Returns
    -------
    enhanced : np.ndarray  (H, W) uint8, ready for thresholding
    H_raw    : np.ndarray  (H, W) float, raw hematoxylin concentrations
    """
    H_raw, _ = deconvolve_he(image_rgb)

    # Normalise [0, 1] → uint8
    lo, hi = H_raw.min(), H_raw.max()
    H_uint8 = ((H_raw - lo) / (hi - lo + 1e-8) * 255).astype(np.uint8)

    # CLAHE
    clahe   = cv2.createCLAHE(clipLimit=params["clahe_clip_limit"],
                               tileGridSize=params["clahe_tile_size"])
    H_clahe = clahe.apply(H_uint8)

    # Gaussian smoothing
    sigma  = params["gaussian_sigma"]
    ksize  = int(6 * sigma + 1) | 1          # must be odd
    smooth = cv2.GaussianBlur(H_clahe, (ksize, ksize), sigmaX=sigma)

    return smooth, H_raw


## 6  · Segmentation Pipeline

In [ ]:
def segment_nuclei(image_rgb: np.ndarray,
                   params: dict = PARAMS,
                   use_gpu: bool = False) -> np.ndarray:
    """
    Classical nucleus segmentation pipeline.

    Steps
    -----
    1. Preprocessing  (deconvolution + CLAHE + smoothing)
    2. Otsu thresholding with adjustable factor
    3. Morphological opening + closing + hole-filling
    4. Distance transform  (GPU-accelerated if CuPy available)
    5. Local-maxima markers + Watershed
    6. Size filtering

    Parameters
    ----------
    image_rgb : np.ndarray  (H, W, 3) uint8
    params    : dict — see PARAMS cell for tunable knobs
    use_gpu   : bool — use CuPy for distance transform if available

    Returns
    -------
    pred_mask : np.ndarray  (H, W) uint8  — 255 = nucleus, 0 = background
    """
    # 1. Preprocessing
    H_smooth, _ = preprocess(image_rgb, params)

    # 2. Otsu Thresholding
    otsu_val, _ = cv2.threshold(H_smooth, 0, 255,
                                cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    thresh       = float(otsu_val) * params["otsu_factor"]
    binary       = (H_smooth >= thresh).astype(np.uint8) * 255

    # 3. Morphological Cleanup
    r_o = params["open_radius"]; r_c = params["close_radius"]
    k_o = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*r_o+1, 2*r_o+1))
    k_c = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*r_c+1, 2*r_c+1))

    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  k_o)   # remove noise
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k_c)   # fill micro-gaps

    # Fill enclosed holes robustly (scipy)
    binary_bool   = binary > 0
    binary_bool   = binary_fill_holes(binary_bool)
    binary        = (binary_bool * 255).astype(np.uint8)

    # 4. Distance Transform
    if use_gpu and GPU_AVAILABLE:
        # GPU path
        _b_cp  = cp.asarray(binary_bool.astype(np.float32))
        _d_cp  = cpnd.distance_transform_edt(_b_cp)
        dist   = cp.asnumpy(_d_cp)
    else:
        dist = distance_transform_edt(binary_bool)

    dist_max  = dist.max()
    if dist_max < 1e-6:
        return binary          # degenerate, return raw threshold result

    dist_norm = dist / dist_max

    # 5a. Local-Maxima Markers
    thresh_abs = params["dist_thresh_frac"] * dist_norm.max()

    # peak_local_max API: returns coordinate array (N, 2)
    try:
        coords = peak_local_max(
            dist_norm,
            min_distance   = params["peak_min_dist"],
            threshold_abs  = thresh_abs,
            labels         = binary_bool,          # restrict to foreground
        )
    except TypeError:
        # Older scikit-image without 'labels' kwarg
        coords = peak_local_max(
            dist_norm,
            min_distance  = params["peak_min_dist"],
            threshold_abs = thresh_abs,
        )

    if len(coords) == 0:
        return binary          # no peaks → return morphological result

    markers          = np.zeros(binary.shape, dtype=np.int32)
    markers[coords[:, 0], coords[:, 1]] = 1
    markers, _       = ndi.label(markers)           # unique integer per peak

    # 5b. Watershed
    try:
        labels = watershed(-dist_norm, markers,
                           mask=binary_bool, compactness=0.001)
    except TypeError:                               # older skimage
        labels = watershed(-dist_norm, markers, mask=binary_bool)

    pred_mask = (labels > 0).astype(np.uint8) * 255

    # 6. Size Filtering
    lbl_out, n_lbl = ndi.label(pred_mask)
    mn, mx         = params["min_area_px"], params["max_area_px"]
    for i in range(1, n_lbl + 1):
        pix = lbl_out == i
        if pix.sum() < mn or pix.sum() > mx:
            pred_mask[pix] = 0

    return pred_mask


## 7  · Evaluation Metrics

In [ ]:
def compute_metrics(pred: np.ndarray, gt: np.ndarray) -> dict:
    """
    Compute pixel-level segmentation metrics

    Returns dict with:
      IoU       – Jaccard Index  =  TP / (TP + FP + FN)
      Dice      – F1 score       =  2·TP / (2·TP + FP + FN)
      Precision –                =  TP / (TP + FP)
      Recall    –                =  TP / (TP + FN)
      TP, FP, FN, TN  (pixel counts)
    """
    p  = pred.astype(bool)
    g  = gt.astype(bool)
    tp = int(np.logical_and(p,  g).sum())
    fp = int(np.logical_and(p, ~g).sum())
    fn = int(np.logical_and(~p, g).sum())
    tn = int(np.logical_and(~p, ~g).sum())

    denom_iou  = tp + fp + fn
    denom_dice = 2 * tp + fp + fn
    denom_prec = tp + fp
    denom_rec  = tp + fn

    return dict(
        IoU       = tp / denom_iou  if denom_iou  > 0 else 1.0,
        Dice      = (2*tp) / denom_dice if denom_dice > 0 else 1.0,
        Precision = tp / denom_prec if denom_prec > 0 else 1.0,
        Recall    = tp / denom_rec  if denom_rec  > 0 else 1.0,
        TP=tp, FP=fp, FN=fn, TN=tn,
    )


## 8  · Visualisation

In [ ]:
def visualise_result(image_rgb, gt_mask, pred_mask, metrics,
                     title="", save_path=None):
    """6-panel diagnostic figure for one image."""
    iou_s  = f"IoU = {metrics['IoU']:.4f}"
    dice_s = f"Dice = {metrics['Dice']:.4f}"

    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle(f"{title}\n{iou_s}  |  {dice_s}",
                 fontsize=13, fontweight="bold")

    # row 0
    axes[0, 0].imshow(image_rgb);               axes[0, 0].set_title("Original H&E Image")
    axes[0, 1].imshow(gt_mask,   cmap="Greens"); axes[0, 1].set_title("Ground Truth")
    axes[0, 2].imshow(pred_mask, cmap="Oranges");axes[0, 2].set_title("Prediction")

    # row 1  overlay
    def overlay(img, mask, colour):
        ov = img.astype(np.float32).copy()
        ov[mask > 0] = ov[mask > 0] * 0.45 + np.array(colour) * 0.55
        return np.clip(ov, 0, 255).astype(np.uint8)

    axes[1, 0].imshow(overlay(image_rgb, gt_mask,   [0, 220, 0]))
    axes[1, 0].set_title("GT Overlay (green)")

    axes[1, 1].imshow(overlay(image_rgb, pred_mask, [255, 140, 0]))
    axes[1, 1].set_title("Prediction Overlay (orange)")

    # error map
    gt_b = gt_mask > 0;  pr_b = pred_mask > 0
    err  = np.zeros((*gt_mask.shape, 3), dtype=np.uint8)
    err[ np.logical_and( gt_b,  pr_b)] = [0,   210,   0]   # TP – green
    err[ np.logical_and(~gt_b,  pr_b)] = [220,   0,   0]   # FP – red
    err[ np.logical_and( gt_b, ~pr_b)] = [0,     0, 220]   # FN – blue

    axes[1, 2].imshow(err)
    axes[1, 2].legend(
        handles=[mpatches.Patch(color="#00D200", label=f"TP {metrics['TP']:,}"),
                 mpatches.Patch(color="#DC0000", label=f"FP {metrics['FP']:,}"),
                 mpatches.Patch(color="#0000DC", label=f"FN {metrics['FN']:,}")],
        loc="lower right", fontsize=9,
    )
    axes[1, 2].set_title("Error Map  (TP / FP / FN)")

    for ax in axes.flat:
        ax.axis("off")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"    Saved → {save_path}")
    plt.show(); plt.close()


## 9  · Main Processing

In [ ]:
def load_image(img_path: Path) -> np.ndarray:
    """Load TIFF or standard image file.  Always returns uint8 RGB (H, W, 3)"""
    if _USE_TIFFFILE:
        img = tifffile.imread(str(img_path))
    else:
        img = cv2.cvtColor(cv2.imread(str(img_path), cv2.IMREAD_COLOR),
                           cv2.COLOR_BGR2RGB)

    # Normalise channel count
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    elif img.ndim == 3 and img.shape[2] > 3:
        img = img[:, :, :3]

    # Normalise dtype to uint8
    if img.dtype != np.uint8:
        img = img.astype(np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8) * 255
        img = img.astype(np.uint8)

    return img


In [ ]:
#  MAIN LOOP, processes all 4 images
all_results = []
DEVICE_TAG  = "GPU" if GPU_AVAILABLE else "CPU"
print(f"Device : {DEVICE_TAG}\n")

for name in IMAGE_NAMES:
    print("=" * 65)
    print(f"  {name}")
    print("=" * 65)

    img_path = TISSUE_DIR / f"{name}.tif"
    xml_path = ANNOTATION_DIR / f"{name}.xml"

    # Load
    image   = load_image(img_path)
    print(f"    Image shape : {image.shape}   dtype: {image.dtype}")
    gt_mask = parse_xml_to_mask(xml_path, image.shape)

    # Segment
    t0           = time.perf_counter()
    pred_mask    = segment_nuclei(image, params=PARAMS, use_gpu=GPU_AVAILABLE)
    elapsed      = time.perf_counter() - t0

    # Metric
    m = compute_metrics(pred_mask, gt_mask)

    print(f"    IoU  (Jaccard)  : {m['IoU']:.4f}")
    print(f"    Dice Similarity : {m['Dice']:.4f}")
    print(f"    Precision       : {m['Precision']:.4f}")
    print(f"    Recall          : {m['Recall']:.4f}")
    print(f"    Running Time    : {elapsed:.4f} s   [{DEVICE_TAG}]")

    # Visualise
    visualise_result(image, gt_mask, pred_mask, m,
                     title=name,
                     save_path=OUTPUT_DIR / f"{name}_result.png")

    all_results.append({
        "Image"       : name,
        "IoU"         : round(m["IoU"],       4),
        "Dice"        : round(m["Dice"],      4),
        "Precision"   : round(m["Precision"], 4),
        "Recall"      : round(m["Recall"],    4),
        "Running Time": round(elapsed,        4),
        "Device"      : DEVICE_TAG,
    })
    print()


## 10  · Results Summary

In [ ]:
df = pd.DataFrame(all_results)

# Console table
print("\n" + "=" * 72)
print("  FINAL RESULTS SUMMARY")
print("=" * 72)
print(df[["Image", "IoU", "Dice", "Precision", "Recall",
          "Running Time", "Device"]].to_string(index=False))
print("=" * 72)
print(f"  Mean IoU      : {df['IoU'].mean():.4f}  (± {df['IoU'].std():.4f})")
print(f"  Mean Dice     : {df['Dice'].mean():.4f}  (± {df['Dice'].std():.4f})")
print(f"  Mean Time     : {df['Running Time'].mean():.4f} s")
print("=" * 72)

# Save CSV
csv_path = OUTPUT_DIR / "results_summary.csv"
df.to_csv(csv_path, index=False)
print(f"\n  → CSV saved : {csv_path}")

# Google Form values
print("\n" + "═" * 55)
print("  >>> VALUES FOR GOOGLE FORM LEADERBOARD <<<")
print("═" * 55)
for row in all_results:
    print(f"  {row['Image']}")
    print(f"    IoU          : {row['IoU']}")
    print(f"    Dice         : {row['Dice']}")
    print(f"    Running Time : {row['Running Time']} s")
    print()


In [ ]:
# Performance Bar Charts
short = [n.split("-")[1] + "\n" + n.split("-")[2][:5]
         for n in df["Image"]]
colours = ["#2196F3", "#4CAF50", "#FF9800", "#E91E63"]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Nucleus Segmentation — Performance per Image",
             fontsize=13, fontweight="bold")

for ax, col, ylabel in zip(
    axes,
    ["IoU", "Dice", "Running Time"],
    ["IoU (Jaccard Index)", "Dice Similarity Coefficient", "Running Time (s)"],
):
    bars = ax.bar(short, df[col], color=colours, edgecolor="k", linewidth=0.7)
    ax.set_title(ylabel, fontsize=11)
    ax.set_ylim(0, max(df[col]) * 1.28)
    ax.axhline(df[col].mean(), color="red", ls="--", lw=1.5,
               label=f"Mean = {df[col].mean():.3f}")
    ax.legend(fontsize=9)
    for bar, val in zip(bars, df[col]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(df[col]) * 0.015,
                f"{val:.4f}", ha="center", fontsize=8.5, fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "performance_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"→ Plot saved : {OUTPUT_DIR / 'performance_summary.png'}")


## 11  · Tuning Guide [DEPRECATED/NOT IN USE IN UNTIL (TIME): INDEFINITELY]

### Parameter Effects on IoU/Dice

| Symptom | Parameter | Adjustment |
|---|---|---|
| Over-detects (high FP) | `otsu_factor` | ↑ try `0.95 – 1.05` |
| Misses nuclei (high FN) | `otsu_factor` | ↓ try `0.75 – 0.88` |
| Over-splits touching nuclei | `peak_min_dist` | ↑ try `10 – 14` |
| Under-splits (merged nuclei) | `peak_min_dist` | ↓ try `5 – 7` |
| Small noise artifacts remain | `min_area_px` | ↑ try `80 – 150` |
| Small nuclei missed | `min_area_px` | ↓ try `25 – 40` |
| Too few watershed markers | `dist_thresh_frac` | ↓ try `0.18 – 0.25` |
| Too many spurious markers | `dist_thresh_frac` | ↑ try `0.35 – 0.45` |
| Noisy hematoxylin channel | `gaussian_sigma` | ↑ try `1.5 – 2.0` |
| Over-smooth (lost edges) | `gaussian_sigma` | ↓ try `0.5 – 0.8` |

### Expected Performance (classical methods, this dataset)

| Method | Typical IoU | Typical Dice |
|---|---|---|
| Simple Otsu only | 0.40 – 0.52 | 0.57 – 0.68 |
| + Morphology | 0.50 – 0.60 | 0.67 – 0.75 |
| + H&E deconv + Watershed *(this script)* | 0.58 – 0.70 | 0.73 – 0.82 |
| Deep learning (U-Net etc.) | 0.72 – 0.82 | 0.84 – 0.90 |

> **Note:** Results vary per image because the 4 MoNuSeg images come from
> different organs/staining batches (breast, kidney, liver, bladder).
> Images with denser nuclei typically score higher with this pipeline.
